# Airflow - ежедневная агрегация данных

## Цели и задачи проекта

Сервис Яндекс Книги предоставляет доступ к контенту разных форматов, включая текст, аудио и не только. В этом проекте построим в Airflow, который будет запускать PySpark-скрипт для обработки данных и создания витрин. Эти витрины помогут команде сервиса быстрее и проще готовить отчёты.


## Описание данных

Таблица `bookmate.audition` содержит данные об активности пользователей и включает столбцы:

* `audition_id` — уникальный идентификатор сессии чтения или прослушивания;

* `puid` — идентификатор пользователя;

* `usage_platform_ru` — название платформы, с помощью которой пользователь взаимодействует с контентом;

* `msk_business_dt_str` — дата и время события (строка, часовой пояс — МСК);

* `app_version` — версия приложения;

* `adult_content_flg` — значение, которое показывает, был ли контент для взрослых (`True` или `False`);

* `hours` — длительность сессии чтения или прослушивания в часах;

* `hours_sessions_long` — длительность длинных сессий в часах;

* `kids_content_flg` — значение, которое показывает, был ли это детский контент (`True` или `False`);

* `main_content_id` — идентификатор основного контента;

* `usage_geo_id` — идентификатор географического местоположения пользователя.

Таблица `bookmate.content` включает столбцы:

* `main_content_id` — идентификатор основного контента;

* `main_author_id` — идентификатор основного автора контента;

* `main_content_type` — тип контента: аудио, текст или другой;

* `main_content_name` — название контента;

* `main_content_duration_hours` — длительность контента в часах;

* `published_topic_title_list` — список жанров или тем контента.</font>

## Содержимое проекта

1. Написание Spark-кода

2. Создание DAG

3. Запуск Airflow

## 1. Написание Spark-кода

In [ ]:
# filename=my_spark_job.py
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import sys


spark = SparkSession.builder.appName("*").config("*", "*").getOrCreate()

# Указываем порт и параметры кластера ClickHouse
jdbcPort =  '*'
jdbcHostname = "*"
username = "*"
password = "*"
jdbcDatabase = "playground_" + username
jdbcUrl = f"jdbc:clickhouse://{jdbcHostname}:{jdbcPort}/{jdbcDatabase}?ssl=true"

# Получаем аргумент из Airflow
my_date = sys.argv[1].replace('-', '_')

# Считываем исходные данные за нужную дату
df = spark.read.csv(f"*/data_{my_date}/audition_content.csv", inferSchema=True, header=True)

# Строим агрегат по пользователям
result_df = df.groupBy("puid").agg(
    F.countDistinct("audition_id").alias("audition_count"),
    F.avg("hours").alias("avg_hours")
)

result_df.write.format("jdbc") \
    .option("url", jdbcUrl) \
    .option("user", username) \
    .option("password", password) \
    .option("dbtable", "bookmate_user_aggregate") \
    .mode('append') \
    .save()

## 2. Создание DAG

In [ ]:
from datetime import datetime
from airflow import DAG
from airflow.sensors.s3_key_sensor import S3KeySensor
from airflow.providers.yandex.operators.dataproc import DataprocCreatePysparkJobOperator

class PysparkJobOperator(DataprocCreatePysparkJobOperator):
    template_fields = ("cluster_id", "args",)

DAG_ID = "audition_content_analysis"

with DAG(
    DAG_ID,
    description = 'Ежедневный запуск агрегации bookmate',
    schedule='0 16 * * *',
    start_date=datetime(2025, 1, 1),
    tags=["bookmate_user_aggregate"],
    catchup=False
) as dag:
  pass

### Задание 2

In [ ]:
from datetime import datetime
from airflow import DAG
from airflow.sensors.s3_key_sensor import S3KeySensor
from airflow.providers.yandex.operators.dataproc import DataprocCreatePysparkJobOperator

class PysparkJobOperator(DataprocCreatePysparkJobOperator):
    template_fields = ("cluster_id", "args",)

DAG_ID = "audition_content_analysis"

wait_for_input = S3KeySensor(
    task_id='data_s3_sensor',
    poke_interval=60, 
    timeout=3600,
    bucket_name='da-plus-dags',
    bucket_key="script_bookmate/data_{{ ds.replace('-', '_') }}/audition_content.csv",
    mode='poke',
    aws_conn_id='s3',
    wildcard_match=False
)

### Задание 3

In [ ]:
from airflow.providers.yandex.operators.dataproc import DataprocCreatePysparkJobOperator

user = '*'

run_pyspark = PysparkJobOperator(
    task_id="run_pyspark_job",
    name="daily_pyspark_job",
    cluster_id="*",
    args=["{{ ds }}"],
    main_python_file_uri=f"*/{user}/jobs/my_spark_job.py",
)

### Задание 4


In [ ]:
from datetime import datetime
from airflow import DAG
from airflow.sensors.s3_key_sensor import S3KeySensor
from airflow.providers.yandex.operators.dataproc import DataprocCreatePysparkJobOperator

class PysparkJobOperator(DataprocCreatePysparkJobOperator):
    template_fields = ("cluster_id", "args",)

DAG_ID = "audition_content_analysis"

with DAG(
    DAG_ID,
    description = 'Ежедневный запуск агрегации bookmate',
    schedule='0 16 * * *',
    start_date=datetime(2025, 1, 1),
    tags=["bookmate_user_aggregate"],
    catchup=False
) as dag:
    
    user = '*'

    # 1) Ждём появления входного файла в S3
    wait_for_input = S3KeySensor(
        task_id='data_s3_sensor',
        poke_interval=60, 
        timeout=3600,
        bucket_name='da-plus-dags',
        bucket_key="script_bookmate/data_{{ ds.replace('-', '_') }}/audition_content.csv",
        mode='poke',
        aws_conn_id='s3',
        wildcard_match=False
    )

    # 2) Запускаем PySpark-задание на кластере Dataproc (оператор Яндекс Облака)
    run_pyspark = PysparkJobOperator(
        task_id="run_pyspark_job",
        name="daily_pyspark_job",
        cluster_id="*",
        args=["{{ ds }}"],
        main_python_file_uri=f"*/{user}/jobs/my_spark_job.py",
    ) 

    # 3)  Зависимости 
    wait_for_input >> run_pyspark

## 3. Запуск Airflow

Пример записанных данных в БД

```sql
    select *
    from bookmate_user_aggregate
```

|  | puid | audition_count | avg_hours |
|----|------|--------------------|---------------|
| 1 | 682966dc-f9d6 | 1 | 0,09 |
| 2 | 68296740-f9d6 | 4 | 0,36750695 |
| 3 | 6829675e-f9d6 | 1 | 1,3563889 |
| 4 | 682967b8-f9d6 | 2 | 0,28133637 |
| 5 | 682967d6-f9d6 | 1 | 1,09 |
| 6 | 682967f4-f9d6 | 6 | 0,11421296 |
| 7 | 68296830-f9d6 | 2 | 0,0029166667 |
| 8 | 682968a8-f9d6 | 4 | 0,1175 |
| 9 | 682968c6-f9d6 | 2 | 0,14 |
| 10 | 68296902-f9d6 | 4 | 0,072847225 |
